In [1]:
import os
print(os.listdir('/kaggle/input'))

['competitions', 'models']


In [2]:
# ============================================================
# INFERENCE NOTEBOOK — Config
# Edit INFER_THRESH, RADIUS_XY, RADIUS_Z here and re-run Cell 1
# then jump to Cell 5 (no need to reload model) to re-generate submission
# ============================================================
import os, json, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max

# ---- device ----
DEVICE = 'cpu'
if torch.cuda.is_available():
    try:
        _p = nn.Conv2d(1, 1, 3).cuda()
        _ = _p(torch.zeros(1, 1, 8, 8, device='cuda')).cpu()
        del _p
        DEVICE = 'cuda'
    except Exception as e:
        print(f'GPU unusable: {str(e)[:80]} -> CPU')
print(f'Device: {DEVICE}')

# ---- paths ----
COMP_ROOT = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR  = Path(COMP_ROOT) / 'test'
OUT_DIR   = Path('/kaggle/working')
MODELS_DIR = Path('/kaggle/input/models')

# ---- load metadata from unet2d_meta.json if it exists ----
meta_path = next(MODELS_DIR.rglob('*.json'), None)
if meta_path is not None:
    with open(meta_path) as f:
        saved_meta = json.load(f)
    print(f'Meta loaded from {meta_path}:')
    print(saved_meta)
    BASE         = int(saved_meta.get('base',         32))
    POOL         = int(saved_meta.get('pool',          4))
    INFER_THRESH = float(saved_meta.get('detect_thresh', 0.25))
    VAL_RECALL   = saved_meta.get('val_recall', 'unknown')
else:
    # fallback: infer BASE from weight tensor shape
    print('No meta JSON found -- inferring BASE from checkpoint weights')
    _raw = torch.load(next(MODELS_DIR.rglob('*.pt')), map_location='cpu')
    BASE = int(_raw['e1.0.weight'].shape[0])
    POOL = 4          # confirmed from training config
    INFER_THRESH = 0.25
    VAL_RECALL   = 'unknown'
    print(f'Inferred BASE={BASE} from e1.0.weight shape {tuple(_raw["e1.0.weight"].shape)}')

print(f'BASE={BASE} | POOL={POOL} | INFER_THRESH={INFER_THRESH} | val_recall={VAL_RECALL}')

# ---- load checkpoint ----
CKPT_PATH = next(MODELS_DIR.rglob('unet2d_best.pt'),
            next(MODELS_DIR.rglob('*.pt'), None))
if CKPT_PATH is None:
    raise FileNotFoundError(f'No .pt file found under {MODELS_DIR}')
print(f'Loading: {CKPT_PATH}')

# raw state dict -- load directly into model after model is defined in Cell 2
raw_state_dict = torch.load(CKPT_PATH, map_location='cpu')
print(f'State dict keys: {len(raw_state_dict)} tensors')

SCALE = np.array([1.625, 0.40625, 0.40625])
print(f'SCALE={SCALE} | TEST_DIR exists={TEST_DIR.exists()}')

Device: cuda
Meta loaded from /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default/3/unet2d_meta.json:
{'base': 32, 'pool': 4, 'gauss_sigma': 2.0, 'best_thresh': 0.05, 'best_val_recall': 0.6936936936936937, 'min_peak_distance': 5, 'nucleus_diam_um': 8.0, 'clahe_clip': 3.0, 'clahe_grid': 8, 'w_pos': 15.0, 'w_bg': 1.0, 'w_ign': 0.01, 'z_margin': 5, 'neg_slices_per_t': 4, 'epochs_trained': 30}
BASE=32 | POOL=4 | INFER_THRESH=0.25 | val_recall=unknown
Loading: /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default/3/unet2d_best.pt
State dict keys: 76 tensors
SCALE=[1.625   0.40625 0.40625] | TEST_DIR exists=True


In [3]:
# Must match the architecture used in training exactly
# If you changed BASE in the training notebook, update it here too
def _block2d(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
    )

class UNet2D(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.e1   = _block2d(1,   b)
        self.e2   = _block2d(b,   b*2)
        self.pool = nn.MaxPool2d(2)
        self.bn   = _block2d(b*2, b*4)
        self.u2   = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.d2   = _block2d(b*4, b*2)
        self.u1   = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.d1   = _block2d(b*2, b)
        self.out  = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        b  = self.bn(self.pool(e2))
        d2 = self.d2(torch.cat([self.u2(b),  e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet2D(base=BASE).to(DEVICE)

# raw state dict -- load directly, no unwrapping needed
model.load_state_dict(raw_state_dict)
model.eval()
print(f'UNet2D loaded: {sum(p.numel() for p in model.parameters()):,} params')

with torch.no_grad():
    _t = torch.zeros(1, 1, 64, 64, device=DEVICE)
    _o = model(_t)
    assert _o.shape == _t.shape, f'Shape mismatch: {_o.shape}'
    print(f'Forward OK: {tuple(_t.shape)} -> {tuple(_o.shape)}')
    del _t, _o

UNet2D loaded: 467,233 params
Forward OK: (1, 1, 64, 64) -> (1, 1, 64, 64)


In [4]:
_ZC = {}

def read_meta(zp):
    with open(Path(zp) / '0' / 'zarr.json') as f:
        m = json.load(f)
    return dict(shape=tuple(m['shape']), dtype=np.dtype(m['data_type']))

def load_vol(zp, t, meta=None):
    try:
        import zarr
        k = str(zp)
        if k not in _ZC:
            _ZC[k] = zarr.open(k, mode='r')['0']
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None:
            meta = read_meta(zp)
        buf = blosc2.decompress(
            open(Path(zp) / '0' / 'c' / str(t) / '0' / '0' / '0', 'rb').read()
        )
        return np.frombuffer(buf, dtype=meta['dtype']).reshape(meta['shape'][1:])

def pool_xy(vol, f=POOL):
    Z, Y, X = vol.shape
    Y2, X2 = (Y // f) * f, (X // f) * f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2 // f, f, X2 // f, f).mean(axis=(2, 4))

def normalize_slice_clahe(slc, clip=2.0, grid=8):
    lo = float(np.percentile(slc, 2.0))
    hi = float(np.percentile(slc, 99.0))
    if hi <= lo:
        return np.zeros_like(slc, dtype=np.float32)
    scaled = np.clip((slc - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)
    clahe  = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    return clahe.apply(scaled).astype(np.float32) / 255.0

print('I/O helpers ready.')

I/O helpers ready.


In [5]:
# ---- inference config ----
NUCLEUS_DIAM_UM   = 8.0
VOXEL_XY_POOLED   = 0.40625 * POOL
MIN_PEAK_DISTANCE = max(5, int(NUCLEUS_DIAM_UM / VOXEL_XY_POOLED))
MIN_CONSECUTIVE_Z = 2
NMS_RADIUS_XY     = 3.0
NMS_RADIUS_Z      = 2

# INFER_THRESH already set in Cell 1 from unet2d_meta.json or default 0.25
print(f'INFER_THRESH={INFER_THRESH} | MIN_PEAK_DISTANCE={MIN_PEAK_DISTANCE}')
print(f'MIN_CONSECUTIVE_Z={MIN_CONSECUTIVE_Z}')
print(f'NMS_RADIUS_XY={NMS_RADIUS_XY} | NMS_RADIUS_Z={NMS_RADIUS_Z}')

def detect_volume_with_scores(pvol, model, thresh=INFER_THRESH,
                               min_dist=MIN_PEAK_DISTANCE,
                               min_consec_z=MIN_CONSECUTIVE_Z):
    model.eval()
    Z = pvol.shape[0]
    slice_dets   = {}
    slice_scores = {}

    with torch.no_grad():
        for z in range(Z):
            slc = normalize_slice_clahe(pvol[z])
            xt  = torch.from_numpy(slc[None, None]).to(DEVICE)
            hm  = torch.sigmoid(model(xt))[0, 0].cpu().numpy()
            pk  = peak_local_max(hm, min_distance=min_dist,
                                  threshold_abs=thresh, exclude_border=False)
            slice_dets[z]   = pk
            slice_scores[z] = np.array([hm[p[0], p[1]] for p in pk]) \
                               if len(pk) > 0 else np.array([])

    confirmed = []
    scores    = []
    for z in range(Z):
        pk_z  = slice_dets.get(z,  np.zeros((0, 2)))
        scr_z = slice_scores.get(z, np.array([]))
        if len(pk_z) == 0:
            continue
        for idx, det_yx in enumerate(pk_z):
            consec = 1
            cur_yx = det_yx.copy()
            for dz in range(1, min_consec_z):
                pk_next = slice_dets.get(z + dz, np.zeros((0, 2)))
                if len(pk_next) == 0:
                    break
                dists = np.linalg.norm(pk_next - cur_yx, axis=1)
                ci    = np.argmin(dists)
                if dists[ci] <= NMS_RADIUS_XY * 2:
                    consec += 1
                    cur_yx  = pk_next[ci]
                else:
                    break
            if consec >= min_consec_z:
                confirmed.append((z, float(det_yx[0]), float(det_yx[1])))
                scores.append(float(scr_z[idx]) if idx < len(scr_z) else 0.0)

    return confirmed, scores

def nms_3d(confirmed, scores, radius_xy=NMS_RADIUS_XY, radius_z=NMS_RADIUS_Z):
    if len(confirmed) == 0:
        return []
    dets  = np.array(confirmed, dtype=np.float64)
    scrs  = np.array(scores,    dtype=np.float64)
    order = np.argsort(scrs)[::-1]
    suppressed = np.zeros(len(dets), dtype=bool)
    keep  = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        zi, yi, xi = dets[i]
        dz  = np.abs(dets[:, 0] - zi)
        dxy = np.sqrt((dets[:, 1] - yi)**2 + (dets[:, 2] - xi)**2)
        suppressed[(dz <= radius_z) & (dxy <= radius_xy)] = True
    return [tuple(dets[i]) for i in keep]

print('Detection functions ready.')

INFER_THRESH=0.25 | MIN_PEAK_DISTANCE=5
MIN_CONSECUTIVE_Z=2
NMS_RADIUS_XY=3.0 | NMS_RADIUS_Z=2
Detection functions ready.


In [6]:
# ============================================================
# CELL 5 — DIVISION-AWARE LINKER (improved)
# Re-run if you change any of the distance parameters below.
# ============================================================

MAX_LINK_DIST_UM = 15.0   # regular frame-to-frame linking distance
DIV_MAX_DIST_UM  = 8.0    # TIGHTER threshold for division assignment
                           # was using MAX_LINK_DIST_UM (15um) before --
                           # too permissive, caused false divisions at t=0->1
                           # real parent-daughter distances measured: mean 6.3um, max 11.8um
                           # 8.0um covers most real divisions while rejecting noise
MIN_DAUGHTER_DIST_UM = 3.0 # daughters must be at least this far apart (physical um)
                            # if two "daughters" are <3um apart they're likely
                            # two detections of the same nucleus, not a real division

def link_timepoints(prev_nodes, curr_nodes,
                    max_dist_um=MAX_LINK_DIST_UM,
                    div_max_dist_um=DIV_MAX_DIST_UM,
                    min_daughter_dist_um=MIN_DAUGHTER_DIST_UM):
    """
    Division-aware nearest-neighbour linker.
    Two-stage:
      Stage 1 — standard 1-to-1 optimal assignment (linear_sum_assignment)
      Stage 2 — division check: unmatched curr nodes that are
                close to an already-matched prev node AND sufficiently
                far from that prev node's first daughter get a second edge.
    """
    prev_ids = list(prev_nodes.keys())
    curr_ids = list(curr_nodes.keys())
    if not prev_ids or not curr_ids:
        return []

    prev_phys = np.array([prev_nodes[i] for i in prev_ids]) * SCALE
    curr_phys = np.array([curr_nodes[i] for i in curr_ids]) * SCALE
    dist = np.sqrt(((prev_phys[:,None] - curr_phys[None,:])**2).sum(2))

    # Stage 1: standard 1-to-1 assignment
    row_ind, col_ind = linear_sum_assignment(dist)
    edges        = []
    matched_curr = set()
    prev_to_curr = {}  # prev_local_idx -> [list of matched curr_local_idx]

    for ri, ci in zip(row_ind, col_ind):
        if dist[ri, ci] <= max_dist_um:
            edges.append((prev_ids[ri], curr_ids[ci]))
            matched_curr.add(ci)
            prev_to_curr.setdefault(ri, []).append(ci)

    # Stage 2: division check with tighter constraints
    for ci in range(len(curr_ids)):
        if ci in matched_curr:
            continue   # already linked as a regular continuation

        best_ri, best_d = None, div_max_dist_um

        for ri in range(len(prev_ids)):
            # only consider prev nodes that already have exactly 1 match
            # (can't divide into 3+ daughters with this simple linker)
            if ri not in prev_to_curr or len(prev_to_curr[ri]) != 1:
                continue

            if dist[ri, ci] > div_max_dist_um:
                continue   # too far for a real division

            # check that the two daughters are sufficiently far apart
            # in physical space -- if they're very close, it's more likely
            # two detections of the same nucleus than a real cell division
            existing_ci  = prev_to_curr[ri][0]
            daughter_dist_um = np.linalg.norm(
                curr_phys[ci] - curr_phys[existing_ci]
            )
            if daughter_dist_um < min_daughter_dist_um:
                continue   # daughters too close -- likely same nucleus

            if dist[ri, ci] < best_d:
                best_d  = dist[ri, ci]
                best_ri = ri

        if best_ri is not None:
            edges.append((prev_ids[best_ri], curr_ids[ci]))
            prev_to_curr[best_ri].append(ci)

    return edges

print(f'Cell 5 complete.')
print(f'  Regular linking:  max_dist={MAX_LINK_DIST_UM}um')
print(f'  Division linking: max_dist={DIV_MAX_DIST_UM}um, '
      f'min_daughter_sep={MIN_DAUGHTER_DIST_UM}um')

Cell 5 complete.
  Regular linking:  max_dist=15.0um
  Division linking: max_dist=8.0um, min_daughter_sep=3.0um


In [7]:
# ============================================================
# CELL 5b — GAP CLOSING
# Bridges tracks that were broken by a missed detection at one timepoint.
# Re-run if you change MAX_GAP or GAP_MAX_DIST_UM.
# ============================================================

MAX_GAP          = 2      # maximum timepoints to bridge over
                           # 2 means: if a nucleus is missed at t=5,
                           # it will still link t=4 -> t=6
GAP_MAX_DIST_UM  = 12.0   # slightly tighter than regular linking since
                           # a real nucleus shouldn't drift far in 2 frames

def close_track_gaps(all_node_ids, all_nuclei, initial_edges,
                     max_gap=MAX_GAP, gap_max_dist=GAP_MAX_DIST_UM,
                     scale=SCALE):
    """
    After standard frame-to-frame linking, bridges gaps where a nucleus
    was missed for up to max_gap consecutive timepoints.

    all_node_ids: dict {t: [node_id, ...]}
    all_nuclei:   dict {t: [(z,y,x), ...]} -- same order as all_node_ids[t]
    initial_edges: dict {t: [(src_id, tgt_id), ...]}

    Returns new_edges dict with gap-bridging edges added.
    """
    import copy
    new_edges = copy.deepcopy(initial_edges)
    sorted_ts = sorted(all_nuclei.keys())

    # rebuild source/target sets from initial edges
    all_src = set()
    all_tgt = set()
    for edges in new_edges.values():
        for src, tgt in edges:
            all_src.add(src)
            all_tgt.add(tgt)

    for t_idx, t in enumerate(sorted_ts):
        nodes_t  = all_node_ids.get(t, [])
        nuclei_t = all_nuclei.get(t, [])

        # track ends at t: nodes with no outgoing edge
        track_ends     = [nid for nid in nodes_t if nid not in all_src]
        track_end_pos  = [nuclei_t[nodes_t.index(nid)] for nid in track_ends]
        if not track_ends:
            continue

        for gap in range(2, max_gap + 2):
            t_future = t + gap
            if t_future not in all_nuclei:
                continue

            nodes_f  = all_node_ids.get(t_future, [])
            nuclei_f = all_nuclei.get(t_future, [])

            # track starts at t_future: nodes with no incoming edge
            track_starts    = [nid for nid in nodes_f if nid not in all_tgt]
            track_start_pos = [nuclei_f[nodes_f.index(nid)]
                                for nid in track_starts]
            if not track_starts:
                continue

            end_phys   = np.array(track_end_pos,   dtype=np.float64) * scale
            start_phys = np.array(track_start_pos, dtype=np.float64) * scale

            dist = np.sqrt(
                ((end_phys[:,None] - start_phys[None,:])**2).sum(2)
            )
            row_ind, col_ind = linear_sum_assignment(dist)

            for ri, ci in zip(row_ind, col_ind):
                if dist[ri, ci] <= gap_max_dist:
                    src_id = track_ends[ri]
                    tgt_id = track_starts[ci]
                    new_edges[t].append((src_id, tgt_id))
                    all_src.add(src_id)
                    all_tgt.add(tgt_id)

    n_gap_edges = sum(len(v) for v in new_edges.values()) - \
                  sum(len(v) for v in initial_edges.values())
    print(f'Gap closing added {n_gap_edges} bridging edges '
          f'(max_gap={max_gap}, max_dist={gap_max_dist}um)')
    return new_edges

print('Cell 5b complete. Gap closing defined.')
print(f'  MAX_GAP={MAX_GAP} timepoints | GAP_MAX_DIST_UM={GAP_MAX_DIST_UM}um')

Cell 5b complete. Gap closing defined.
  MAX_GAP=2 timepoints | GAP_MAX_DIST_UM=12.0um


In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

test_names = sorted(
    d.replace('.zarr', '')
    for d in os.listdir(TEST_DIR)
    if d.endswith('.zarr')
)
print(f'Test datasets: {len(test_names)}')
for n in test_names:
    print(f'  {n}')

all_rows = []

for folder_name in test_names:
    zp   = str(TEST_DIR / f'{folder_name}.zarr')
    meta = read_meta(zp)
    n_t  = meta['shape'][0]
    print(f'\n{folder_name} | {n_t} timepoints | shape {meta["shape"]}')

    # ---- detection ----
    # all_cents_orig: list of np.array (N,3) in original voxel coords, one per t
    # nuclei_by_t / node_ids_by_t: dicts for gap closing
    all_cents_orig = []
    nuclei_by_t    = {}
    node_ids_by_t  = {}
    global_nid     = 1     # globally unique node id, resets per dataset

    for t in range(n_t):
        vol  = load_vol(zp, t, meta)
        pvol = pool_xy(vol)

        confirmed, scores = detect_volume_with_scores(pvol, model)
        nuclei            = nms_3d(confirmed, scores)

        if len(nuclei) == 0:
            all_cents_orig.append(np.zeros((0, 3), dtype=np.float64))
            nuclei_by_t[t]   = []
            node_ids_by_t[t] = []
        else:
            arr  = np.array(nuclei, dtype=np.float64)
            orig = np.stack([
                arr[:, 0],
                arr[:, 1] * POOL,
                arr[:, 2] * POOL,
            ], axis=1)
            all_cents_orig.append(orig)

            # assign node ids for this timepoint
            t_nids = list(range(global_nid, global_nid + len(nuclei)))
            global_nid += len(nuclei)
            node_ids_by_t[t] = t_nids
            nuclei_by_t[t]   = [tuple(orig[i]) for i in range(len(nuclei))]

        if t % 20 == 0 or t == n_t - 1:
            print(f'  t={t:3d}/{n_t-1} | {len(nuclei)} nuclei detected')

    # ---- linking: frame-to-frame with division awareness ----
    # link_timepoints expects {node_id: (z,y,x)} dicts for prev and curr
    node_rows  = []
    edge_rows  = []
    edges_by_t = {}    # {t: [(src_nid, tgt_nid)]} for gap closing

    for t in range(n_t):
        # build node rows
        for i, nid in enumerate(node_ids_by_t.get(t, [])):
            z, y, x = nuclei_by_t[t][i]
            node_rows.append({
                'dataset':   folder_name,
                'row_type':  'node',
                'node_id':   nid,
                't':         t,
                'z':         int(round(z)),
                'y':         int(round(y)),
                'x':         int(round(x)),
                'source_id': -1,
                'target_id': -1,
            })

        if t == 0:
            continue

        # build {nid: (z,y,x)} dicts for linker
        prev_t    = t - 1
        prev_nids = node_ids_by_t.get(prev_t, [])
        curr_nids = node_ids_by_t.get(t,      [])
        prev_nodes = {nid: nuclei_by_t[prev_t][i]
                      for i, nid in enumerate(prev_nids)}
        curr_nodes = {nid: nuclei_by_t[t][i]
                      for i, nid in enumerate(curr_nids)}

        frame_edges = link_timepoints(prev_nodes, curr_nodes)
        edges_by_t[prev_t] = frame_edges

        for src, tgt in frame_edges:
            edge_rows.append({
                'dataset':   folder_name,
                'row_type':  'edge',
                'node_id':   -1,
                't':         -1,
                'z':         -1,
                'y':         -1,
                'x':         -1,
                'source_id': src,
                'target_id': tgt,
            })

    # ---- gap closing ----
    closed_edges = close_track_gaps(node_ids_by_t, nuclei_by_t, edges_by_t)

    # add gap-bridging edges that weren't in the original frame-to-frame pass
    original_edge_pairs = {(r['source_id'], r['target_id']) for r in edge_rows}
    for t_edges in closed_edges.values():
        for src, tgt in t_edges:
            if (src, tgt) not in original_edge_pairs:
                edge_rows.append({
                    'dataset':   folder_name,
                    'row_type':  'edge',
                    'node_id':   -1,
                    't':         -1,
                    'z':         -1,
                    'y':         -1,
                    'x':         -1,
                    'source_id': src,
                    'target_id': tgt,
                })

    # ---- short track filter ----
    # union-find: keep only tracks with >= 3 nodes
    parent = {r['node_id']: r['node_id'] for r in node_rows}

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a

    def union(a, b):
        parent[find(a)] = find(b)

    for r in edge_rows:
        if r['source_id'] in parent and r['target_id'] in parent:
            union(r['source_id'], r['target_id'])

    comp = defaultdict(list)
    for r in node_rows:
        comp[find(r['node_id'])].append(r['node_id'])
    keep = set()
    for members in comp.values():
        if len(members) >= 3:
            keep.update(members)

    node_rows = [r for r in node_rows if r['node_id'] in keep]
    edge_rows = [r for r in edge_rows
                 if r['source_id'] in keep and r['target_id'] in keep]

    n_nodes = len(node_rows)
    n_edges = len(edge_rows)
    print(f'  -> kept: {n_nodes} nodes | {n_edges} edges')

    all_rows.extend(node_rows)
    all_rows.extend(edge_rows)

# ---- write submission ----
submission = pd.DataFrame(
    all_rows,
    columns=['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
             'source_id', 'target_id']
)
submission.index.name = 'id'
sub_path = OUT_DIR / 'submission.csv'
submission.to_csv(sub_path)

print(f'\n{"="*50}')
print(f'submission.csv written: {sub_path}')
print(f'  Total rows : {len(submission)}')
print(f'  Node rows  : {(submission.row_type == "node").sum()}')
print(f'  Edge rows  : {(submission.row_type == "edge").sum()}')
print(f'  Datasets   : {submission.dataset.nunique()}')

assert set(submission.columns) == {
    'dataset','row_type','node_id','t','z','y','x','source_id','target_id'}
assert submission.index.name == 'id'
assert submission[submission.row_type=='node']['source_id'].eq(-1).all()
assert submission[submission.row_type=='node']['target_id'].eq(-1).all()
assert submission[submission.row_type=='edge']['node_id'].eq(-1).all()
print('Format checks passed.')
print(f'{"="*50}')

# ---- stats plot ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ds in submission.dataset.unique():
    nodes_ds = submission[(submission.dataset==ds) & (submission.row_type=='node')]
    counts   = nodes_ds.groupby('t').size()
    axes[0].plot(counts.index, counts.values, label=ds, alpha=0.8)
axes[0].set_xlabel('Timepoint'); axes[0].set_ylabel('Nodes')
axes[0].set_title('Nodes per timepoint'); axes[0].legend(fontsize=7)

node_counts = submission[submission.row_type=='node'].groupby(['dataset','t']).size()
axes[1].hist(node_counts.values, bins=30)
axes[1].set_xlabel('Nodes per timepoint'); axes[1].set_title('Distribution')

edge_counts = submission[submission.row_type=='edge'].groupby('dataset').size()
axes[2].bar(range(len(edge_counts)), edge_counts.values)
axes[2].set_xticks(range(len(edge_counts)))
axes[2].set_xticklabels(edge_counts.index, rotation=20, ha='right', fontsize=7)
axes[2].set_title('Edges per dataset')

plt.tight_layout()
plt.savefig(OUT_DIR / 'submission_stats.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'Stats plot saved.')

Test datasets: 4
  44b6_0113de3b
  44b6_0b24845f
  6bba_05b6850b
  6bba_05db0fb1

44b6_0113de3b | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 228 nuclei detected
  t= 20/99 | 237 nuclei detected
  t= 40/99 | 245 nuclei detected
  t= 60/99 | 268 nuclei detected
  t= 80/99 | 252 nuclei detected
  t= 99/99 | 237 nuclei detected
Gap closing added 238 bridging edges (max_gap=2, max_dist=12.0um)
  -> kept: 24348 nodes | 23896 edges

44b6_0b24845f | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 426 nuclei detected
  t= 20/99 | 346 nuclei detected
  t= 40/99 | 305 nuclei detected
  t= 60/99 | 378 nuclei detected
  t= 80/99 | 419 nuclei detected
  t= 99/99 | 392 nuclei detected
Gap closing added 650 bridging edges (max_gap=2, max_dist=12.0um)
  -> kept: 36417 nodes | 35429 edges

6bba_05b6850b | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 145 nuclei detected
  t= 20/99 | 134 nuclei detected
  t= 40/99 | 131 nuclei detected
  t= 60/99 | 134 nuclei detected
  t= 